# Lesson 6, Part 3

Integrals depend on cumulative sums, while derivatives depend on differences between neighboring elements, 
so it is helpful to have functions that compute cumulative sums and neighoring differences. 
In numpy, the first is called **cumsum** and the second is called **diff**. 
Note that **diff** reduces the length of a series by 1.

In [ ]:
# initializations
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8")  # insert plt.style.use("default") to convert back      

def cum_sum(x):
    cumsum = np.zeros(len(x))
    running_total = 0
    for n in range(len(x)):
        running_total += x[n]
        cumsum[n] = running_total
    return cumsum

def diff_prev(x):
    return x[1:] - x[:-1]
        
# demo
x = np.arange(1,15,1)
y1 = cum_sum(x)
y2 = np.cumsum(x)
x_shift1 = diff_prev(y1)
x_shift2 = np.diff(y2)

print(' x  cum_sum np.cumsum diff_prev np.diff')
for n in range(len(x)-1):
    print('{0:2}'.format(x[n]),'{0:7.0f}'.format(y1[n]),'{0:7}'.format(y2[n]),'{0:8.0f}'.format(x_shift1[n]),'{0:8}'.format(x_shift2[n])) 
print('{0:2}'.format(x[-1]),'{0:7.0f}'.format(y1[-1]),'{0:7}'.format(y2[-1]))

We can use these functions to estimate integrals and derivatives. 
We'll test the estimators on exp(*x*), where ideally they should leave values unchanged.

In [ ]:
def integral_approx(y,dx):
    avg_y = 0.5 * (y[:-1] + y[1:])
    cumsum_avg_y = np.cumsum(avg_y)
    integral = np.zeros(len(y))
    integral[1:] = cumsum_avg_y * dx 
    return integral

def deriv_approx(y,dx):
    dy = np.zeros(len(y)+1)
    dy[1:-1] = np.diff(y)
    dy[0] = 2 * dy[1] - dy[2]  # assume linear at end 
    dy[-1] = 2 * dy[-2] - dy[-3] 
    deriv = (dy[:-1] + dy[1:]) / (2 * dx)
    return deriv

x = np.linspace(0,5,1001)
dx = (x[-1] - x[1]) / len(x)
y = np.exp(x)
y_integral = integral_approx(y,dx) + 1  # add 1 because exp(0)=1
y_deriv = deriv_approx(y,dx)
rel_diverge_int = np.abs(np.log(y_integral/y))
mean_rd = np.mean(rel_diverge_int)
max_rd = np.max(rel_diverge_int)
print('Relative divergence of integral_approx(exp(x)) averages','{:.4f}'.format(mean_rd),'with maximum','{:.4f}'.format(max_rd))
rel_diverge_deriv = np.abs(np.log(y_deriv/y))
mean_rd = np.mean(rel_diverge_deriv)
max_rd = np.max(rel_diverge_deriv)
print('Relative divergence of deriv_approx(exp(x)) averages','{:.4f}'.format(mean_rd),'with maximum','{:.4f}'.format(max_rd))

Let's use integral_approx to track the boosts in acceleration, velocity, 
and distance of an ideal rocket in empty space. 
The code here assumes that fuel is expelled at constant rate until depleted.

In [ ]:
def rocket_ideal(payload_frac,rate_expel,exhaust_speed,num=1000):
    time_expel = (1 - payload_frac) / rate_expel
    time = np.linspace(0,time_expel,num)
    dt = time_expel / (num - 1)
    mass_rocket = 1 - rate_expel * time     # mass at time t
    exhaust_force = exhaust_speed * rate_expel
    accel = exhaust_force / mass_rocket  
    velocity = integral_approx(accel,dt)    
    distance = integral_approx(velocity,dt)
    target_vel = exhaust_speed * np.log(1/payload_frac)
    return time,accel,velocity,distance,target_vel

payload,expel_rate,exhaust_speed = 0.08,0.002,4.5
time,accel,velocity,distance,target_vel = rocket_ideal(payload,expel_rate,exhaust_speed)
print('Rocket takes','{:.0f}'.format(time[-1]),'seconds to accelerate to','{:.2f}'.format(velocity[-1]),'km/sec')
rel_diverge = np.log(velocity[-1]/target_vel)
rd_print = '{:.1e}'.format(rel_diverge)  # prints in exponential format with 1 digit to right of decimal point
print('Relative divergence in final velocity between computed value and target is',rd_print)
print('Additional distance traveled is','{:.0f}'.format(distance[-1]),'km')

With a constant exhaust rate, acceleration is highly bunched toward the end. 
This could be a problem for people on board. 
Few humans can handle 5g forces for more than two minutes, 
and 9g forces can be deadly even for trained astronauts. 
Moreover, the sudden cessation of acceleration will impose a huge jerk.

In [ ]:
payload,expel_rate,exhaust_speed = 0.08,0.002,4.5
time,accel,velocity,distance,target_vel = rocket_ideal(payload,expel_rate,exhaust_speed)

gravity_earth = 0.009807  # kilometers per second per second
g_force = accel / gravity_earth
print('g-force exceeds 5 for',sum(g_force>5),'seconds and exceeds 8 for',sum(g_force>8),'seconds')

plt.title('Acceleration in g-force with constant exhaust rate')
plt.plot(time,g_force)
plt.xlabel('Time in Seconds')
plt.ylabel('g-force')
plt.show()

As an alternative, let's consider fuel expelled at a constant fractional rate *r*. 
The residual fraction *y* at time *t* is exp(*-rt*), or *t* = -ln(*y/r*).

In [ ]:

plt.title('Plots of exponential decay')
plt.xlabel('Time')
plt.ylabel('Residual Fraction')
t = np.linspace(0,500,200)
for rate in (0.08,0.04,0.02,0.01):
    plt.plot(t,np.exp(-rate*t),label='decay rate '+str(rate))
plt.legend()    

plt.figure()
plt.title('Plots of exponential decay in log terms')
plt.xlabel('Time')
plt.ylabel('Residual Fraction')
for rate in (0.08,0.04,0.02,0.01):
    plt.semilogy(t,np.exp(-rate*t),label='decay rate '+str(rate))
plt.legend()    
plt.show()

Here is the code revised to handle exponentially declining exhaust rates. 
The resulting acceleration is much easier for humans to handle.

In [ ]:
def rocket_ideal_alt(payload_frac,frac_rate_expel,exhaust_speed,residual_frac=0.001,num=1000):
    gravity_earth = 0.009807
    tmax = -np.log(residual_frac) / frac_rate_expel
    time = np.linspace(0,tmax,num)
    dt = tmax / (num - 1)
    mass_fuel = (1 - payload_frac) * np.exp(-frac_rate_expel * time)
    rate_expel = frac_rate_expel * mass_fuel
    exhaust_force = exhaust_speed * rate_expel
    mass_rocket = payload_frac + mass_fuel
    accel = exhaust_force / mass_rocket
    velocity = integral_approx(accel,dt)
    distance = integral_approx(velocity,dt)
    target_vel = exhaust_speed * np.log(1/payload_frac)
    return time,accel,velocity,distance,target_vel

payload,frac_rate_expel,exhaust_speed = 0.08,0.01,4.5
time,accel,velocity,distance,target_vel = rocket_ideal_alt(payload,frac_rate_expel,exhaust_speed)
rel_diverge = np.log(velocity[-1]/target_vel)
print('Rocket takes','{:.0f}'.format(time[-1]),'seconds to accelerate to','{:.2f}'.format(velocity[-1]),'km/sec')
print('Relative divergence in final velocity between computed value and target is','{:.1e}'.format(rel_diverge))
print('Additional distance traveled is','{:.0f}'.format(distance[-1]),'km')
plt.title('Acceleration in g-force for exponentially decaying exhaust')
plt.xlabel('Time in seconds')
plt.ylabel('g-force')
g_force = accel / gravity_earth
plt.plot(time,g_force)
plt.show()

Let's plot the extra velocity and distance traveled for the two specifications, 
after extending the first time span to match the second.

In [ ]:
payload,expel_rate,frac_rate_expel,exhaust_speed = 0.08,0.002,0.01,4.5
time,accel,velocity,distance,target_vel = rocket_ideal(payload,expel_rate,exhaust_speed)
time1,accel1,velocity1,distance1,target_vel1 = rocket_ideal_alt(payload,frac_rate_expel,exhaust_speed)
time_extend = np.append(time,time1[-1]) # add last time1 value to time series
velocity_extend = np.append(velocity,velocity[-1])
distance_last = distance[-1] + (time1[-1] - time[-1]) * velocity[-1]
distance_extend = np.append(distance,distance_last)
plt.subplot(2,1,1)
plt.title('Velocity and Distance')
plt.plot(time_extend,velocity_extend,time1,velocity1)
plt.ylabel('Velocity in km/sec')
plt.legend(labels=('constant exhaust rate','exponentially decaying exhaust rate'))
plt.subplot(2,1,2)
plt.ylabel('Distance in 1000 km')
plt.xlabel('Time in seconds')
plt.plot(time_extend,distance_extend/1000,time1,distance1/1000)
plt.show()

Let's model exponentially decaying exhaust rates with a launch from earth. 
As a first pass, we'll assume the earth's gravity is constant.

In [ ]:
def rocket_earth_alt(payload_frac,frac_rate_expel,exhaust_speed,residual_frac=0.001,num=1000):
    gravity_earth = 0.009807 
    tmax = -np.log(residual_frac) / frac_rate_expel
    time = np.linspace(0,tmax,num)
    dt = tmax / (num - 1)
    mass_fuel = (1 - payload_frac) * np.exp(-frac_rate_expel * time)
    rate_expel = frac_rate_expel * mass_fuel
    exhaust_force = exhaust_speed * rate_expel
    mass_rocket = payload_frac + mass_fuel
    accel = exhaust_force / mass_rocket - gravity_earth
    velocity = integral_approx(accel,dt)
    distance = integral_approx(velocity,dt)
    return time,accel,velocity,distance

payload,frac_rate_expel,exhaust_speed = 0.08,0.01,4.5
time,accel,velocity,distance = rocket_earth_alt(payload,frac_rate_expel,exhaust_speed)
print('Rocket takes','{:.0f}'.format(time[-1]),'seconds to accelerate to','{:.2f}'.format(velocity[-1]),'km/sec')
print('Additional distance traveled is','{:.0f}'.format(distance[-1]),'km')
g_force = accel / gravity_earth
plt.title('Acceleration in g-force for exponentially decaying exhaust')
plt.plot(time,g_force,label='Acceleration from earth')
plt.legend()
plt.show()

Since the earth's gravitational pull will more than halve before all fuel is burned,
we need to amend the code to incorporate the shrinkage.

In [ ]:
def rocket_earth_better(payload_frac,frac_rate_expel,exhaust_speed,residual_frac=0.001,num=1000):
    radius_earth = 6371 # kilometers
    tmax = -np.log(residual_frac) / frac_rate_expel
    time = np.linspace(0,tmax,num)
    dt = tmax / (num - 1)
    mass_fuel = (1 - payload_frac) * np.exp(-frac_rate_expel * time)
    rate_expel = frac_rate_expel * mass_fuel
    exhaust_force = exhaust_speed * rate_expel
    mass_rocket = payload_frac + mass_fuel
    accel = exhaust_force / mass_rocket # baseline absent gravity
    curr_dist = curr_vel = 0
    distance,velocity = np.zeros(num),np.zeros(num)  # need separate specs to distinguish the two
    for i in range(num):
        rel_distance_center = 1 + curr_dist / radius_earth
        gravity = gravity_earth / rel_distance_center**2
        accel[i] -= gravity  # adjust for gravity
        curr_vel += 0.5 * (accel[i] + accel[i-1]) * dt
        velocity[i] = curr_vel
        curr_dist +=  0.5 * (curr_vel + velocity[i-1]) * dt
        distance[i] = curr_dist
    return time,accel,velocity,distance

payload,frac_rate_expel,exhaust_speed,gravity_earth = 0.08,0.01,4.5,0.009807
time,accel,velocity,distance = rocket_earth_better(payload,frac_rate_expel,exhaust_speed)
print('Rocket takes','{:.0f}'.format(time[-1]),'seconds to accelerate to','{:.2f}'.format(velocity[-1]),'km/sec')
print('Additional distance traveled is','{:.0f}'.format(distance[-1]),'km\n')
g_force = accel / gravity_earth
plt.subplot(3,1,1)
plt.title('Acceleration, Velocity, and Distance for Exponentially Decaying Exhaust')
plt.plot(time,g_force)
plt.ylabel('Acceleration in g')
plt.subplot(3,1,2)
plt.plot(time,velocity)
plt.ylabel('Velocity in km/sec')
plt.subplot(3,1,3)
plt.ylabel('Distance in 1000 km')
plt.xlabel('Time in seconds')
plt.plot(time,distance/1000)
plt.show()

Let's now examine air drag on an object falling with velocity *v*. The text considered two possible equations of motion. 
The first was *v' = g - bv* for gravity *g*, positive constant *b*, and initial velocity *v0*, 
with solution *v*(*t*) = *g/b* + (*v0 - g/b*) exp(*-bt*). 
This implies acceleration *a*(*t*) = (*g - b v0*) exp(*-bt*) and displacement *y*(*t*) = *gt/b* + (*v0 - g/b*) (1 -exp(*-bt*)/*b*. 
Let's check that in simulations.

In [ ]:
def air_drag_linear(b_drag,v0=0,num=1000):
    gravity = 9.807  # meters per second per second
    time = np.linspace(0,30,num)
    dt = time[-1] / (num - 1)
    # estimated values
    velocity = np.zeros(num)
    velocity[0] = curr_vel = v0
    for i in range(1,num):
        dv_approx = (gravity - b_drag * curr_vel) * dt
        avg_vel = curr_vel + 0.5 * dv_approx
        curr_vel +=  (gravity - b_drag * avg_vel) * dt
        velocity[i] = curr_vel
    accel = deriv_approx(velocity,dt)
    distance = integral_approx(velocity,dt)
    # exact values
    a0 = gravity - b_drag * v0
    a_exact = a0 * np.exp(-b_drag * time)
    v_exact = gravity / b_drag - a_exact / b_drag
    d_exact = (gravity / b_drag) * time + a_exact / b_drag**2 - a0 / b_drag**2
    return time,accel,velocity,distance,a_exact,v_exact,d_exact

b_drag = 0.2 
time,accel,velocity,distance,a_exact,v_exact,d_exact = air_drag_linear(b_drag)
div_v = np.log(velocity[-1]/v_exact[-1])
div_d = np.log(distance[-1]/d_exact[-1])
print('Relative divergences at end are\n','{:.1e}'.format(div_v),'for velocity and','{:.1e}'.format(div_d),'for distance\n')
plt.title('Divergences between computed and exact values')
plt.plot(time,accel-a_exact,time,velocity-v_exact,time,distance-d_exact)
plt.legend(labels=('Acceleration','Velocity','Distance'))
plt.show()

Let's plot acceleration, velocity and distance for linear drag.

In [ ]:
def plotAccelVelDist(time,accel,velocity,distance,title):
    plt.subplot(3,1,1)
    plt.title(title)
    plt.plot(time,accel,label='accel')
    plt.ylabel('Acceleration')
    plt.subplot(3,1,2)
    plt.plot(time,velocity)
    plt.ylabel('Velocity m/sec')
    plt.subplot(3,1,3)
    plt.plot(time,distance)
    plt.ylabel('Distance meters')
    plt.xlabel("Time in seconds")
    plt.show()

b_drag = 0.2
time,accel,velocity,distance,a_exact,v_exact,d_exact = air_drag_linear(b_drag)
title = 'Acceleration, Velocity, and Distance for linear drag'
plotAccelVelDist(time,accel,velocity,distance,title)

Another equation of air drag presumes quadratic drag *v*' = *g*(1 - *c*<sup>2</sup>*v*<sup>2</sup>). 
When *v*0 = 0 this has solution *v*(*t*) = exp(2*cgt*) -1)/*c*/(exp(2*cgt*) + 1). 
Let's check that too and plot the resulting acceleration, velocity, and distance.

In [ ]:
def air_drag_quadratic(c_drag,num=1000):
    gravity = 9.807  # meters per second
    time = np.linspace(0,30,num)
    # estimated values
    dt = time[-1] / (num - 1)
    velocity = np.zeros(num)
    curr_vel = 0
    for i in range(1,num):
        dv_approx = gravity * (1 - (c_drag * curr_vel)**2) * dt
        avg_vel = curr_vel + 0.5 * dv_approx
        curr_vel +=  gravity * (1 - (c_drag * avg_vel)**2) * dt
        velocity[i] = curr_vel
    accel = deriv_approx(velocity,dt)
    distance = integral_approx(velocity,dt)
    # exact values
    exp_series = np.exp(2 * c_drag * gravity * time)
    v_exact = (exp_series - 1) / c_drag / (exp_series + 1)
    return time,accel,velocity,distance,v_exact

c_drag = 0.02
time,accel,velocity,distance,v_exact = air_drag_quadratic(c_drag)
plt.title('Divergence between computed and exact velocity')
plt.plot(time,velocity-v_exact)
plt.figure()
title = 'Acceleration, Velocity, and Distance for quadratic drag'
plotAccelVelDist(time,accel,velocity,distance,title)